In [12]:
import pandas as pd
import sqlite3
import os

raw = pd.read_csv(
    r"C:\Users\nidhi\Documents\Summer Learning\01-eda-alberta-construction\data\alberta_projects_clean.csv"
)
 
print("=== FLAT FILE SHAPE ===")
print(raw.shape)
 
print("\n=== COLUMNS ===")
print(raw.columns.tolist())
 
print("\n=== FIRST 3 ROWS ===")
print(raw.head(3))
 
print("\n=== COLUMNS THAT REPEAT (candidates for dimension tables) ===")

for col in raw.columns:
    unique = raw[col].nunique()
    print(f"  {col:<25} {unique} unique values")

=== FLAT FILE SHAPE ===
(189, 15)

=== COLUMNS ===
['project_id', 'project_name', 'city', 'province', 'sector', 'contractor', 'project_value_cad', 'start_date', 'end_date', 'status', 'workers_on_site', 'safety_incidents', 'delay_days', 'inspector_name', 'approved']

=== FIRST 3 ROWS ===
     project_id  project_name           city province          sector  \
0  AB-2022-0106    Depot C106     Lethbridge  Alberta      Commercial   
1  AB-2021-0113    Plaza J113  Fort McMurray  Alberta      Commercial   
2  AB-2021-0121  Highway R121       Edmonton  Alberta  Infrastructure   

         contractor  project_value_cad  start_date    end_date     status  \
0  PCL Construction        23838564.61  2022-02-19  2022-01-27  Completed   
1      Fluor Canada        25373009.84  2023-09-17  2023-07-08    On Hold   
2      Ledcor Group        38291677.86  2020-04-19  2020-12-27  Completed   

   workers_on_site  safety_incidents  delay_days inspector_name approved  
0            142.0               5.

DERIVING DIMENSION TABLE

In [13]:
regions = (
    raw[["city", "province"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

regions.insert(0, "region_id", range(1, len(regions) + 1))
 
print("=== REGIONS TABLE DERIVED ===")
print(regions)
print(f"\nTotal regions: {len(regions)}")


=== REGIONS TABLE DERIVED ===
   region_id            city province
0          1      Lethbridge  Alberta
1          2   Fort McMurray  Alberta
2          3        Edmonton  Alberta
3          4        Red Deer  Alberta
4          5    Medicine Hat  Alberta
5          6  Grande Prairie  Alberta
6          7      Blackfalds  Alberta
7          8         Calgary  Alberta

Total regions: 8


In [15]:
contractors = (
    raw[["contractor"]]        
    .drop_duplicates()
    .reset_index(drop=True)
)
 
contractors.insert(0, "contractor_id", range(1, len(contractors) + 1))
contractors = contractors.rename(columns={"contractor": "contractor_name"})
 
print("=== CONTRACTORS TABLE DERIVED ===")
print(contractors)
print(f"\nTotal contractors: {len(contractors)}")

=== CONTRACTORS TABLE DERIVED ===
   contractor_id      contractor_name
0              1     PCL Construction
1              2         Fluor Canada
2              3         Ledcor Group
3              4                 ATCO
4              5    Bird Construction
5              6             EllisDon
6              7              Stantec
7              8  Graham Construction

Total contractors: 8


In [16]:
projects = raw.merge(
    regions[["region_id", "city"]],
    on="city",
    how="left"
)

projects = projects.merge(
    contractors[["contractor_id", "contractor_name"]],
    left_on="contractor",      
    right_on="contractor_name",
    how="left"
)

In [17]:
projects_clean = projects[[
    "project_id",
    "region_id",
    "contractor_id",
    "sector",
    "project_value_cad",
    "start_date",
    "end_date",
    "status",
    "workers_on_site",
    "safety_incidents",
    "delay_days",
    "approved"
]].copy()
 
print("=== PROJECTS FACT TABLE DERIVED ===")
print(projects_clean.shape)
print(projects_clean.head(3))
 

print("\n=== FK NULL CHECK ===")
print(f"Missing region_id:     {projects_clean['region_id'].isnull().sum()}")
print(f"Missing contractor_id: {projects_clean['contractor_id'].isnull().sum()}")

=== PROJECTS FACT TABLE DERIVED ===
(189, 12)
     project_id  region_id  contractor_id          sector  project_value_cad  \
0  AB-2022-0106          1              1      Commercial        23838564.61   
1  AB-2021-0113          2              2      Commercial        25373009.84   
2  AB-2021-0121          3              3  Infrastructure        38291677.86   

   start_date    end_date     status  workers_on_site  safety_incidents  \
0  2022-02-19  2022-01-27  Completed            142.0               5.0   
1  2023-09-17  2023-07-08    On Hold            241.0               5.0   
2  2020-04-19  2020-12-27  Completed            149.0               6.0   

   delay_days approved  
0       113.0       No  
1        48.0      Yes  
2       138.0      Yes  

=== FK NULL CHECK ===
Missing region_id:     0
Missing contractor_id: 0


SQLITE DATABASE AND SCHEMA

In [18]:
os.makedirs("data", exist_ok=True)
 
# Connect — creates the file if it does not exist
conn = sqlite3.connect("data/alberta_projects.db")
cursor = conn.cursor()
 
# Enforce foreign key constraints — SQLite ignores them by default
cursor.execute("PRAGMA foreign_keys = ON")
 
# Drop tables if they exist so we can rerun this cell cleanly
cursor.execute("DROP TABLE IF EXISTS projects")
cursor.execute("DROP TABLE IF EXISTS regions")
cursor.execute("DROP TABLE IF EXISTS contractors")
 
# REGIONS
cursor.execute("""
    CREATE TABLE regions (
        region_id  INTEGER PRIMARY KEY,
        city       TEXT    NOT NULL,
        province   TEXT    NOT NULL
    )
""")
 
# CONTRACTORS
cursor.execute("""
    CREATE TABLE contractors (
        contractor_id   INTEGER PRIMARY KEY,
        contractor_name TEXT    NOT NULL
    )
""")
 
# PROJECTS — references both dimension tables via foreign keys
cursor.execute("""
    CREATE TABLE projects (
        project_id        TEXT    PRIMARY KEY,
        region_id         INTEGER REFERENCES regions(region_id),
        contractor_id     INTEGER REFERENCES contractors(contractor_id),
        sector            TEXT    NOT NULL,
        project_value_cad REAL    CHECK(project_value_cad > 0),
        start_date        TEXT,
        end_date          TEXT,
        status            TEXT,
        workers_on_site   REAL,
        safety_incidents  REAL    DEFAULT 0,
        delay_days        REAL    DEFAULT 0,
        approved          TEXT
    )
""")
 
conn.commit()
print("✓ Schema created — 3 tables ready")
 

✓ Schema created — 3 tables ready


LOAD DERIVED TABLES INTO DATABASE

In [19]:
regions.to_sql("regions",         conn, if_exists="append", index=False)
contractors.to_sql("contractors", conn, if_exists="append", index=False)
projects_clean.to_sql("projects", conn, if_exists="append", index=False)
 
conn.commit()
 
# Verify row counts
print("=== ROW COUNTS AFTER LOAD ===")
for table in ["regions", "contractors", "projects"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<15} {count} rows")

=== ROW COUNTS AFTER LOAD ===
  regions         8 rows
  contractors     8 rows
  projects        189 rows


SQL QUERIES

In [20]:
q1 = pd.read_sql_query("""
    SELECT
        project_id,
        sector,
        ROUND(project_value_cad / 1000000, 2) AS value_millions,
        status,
        delay_days
    FROM projects
    WHERE status = 'Completed'
      AND project_value_cad > 10000000
    ORDER BY project_value_cad DESC
    LIMIT 10
""", conn)
print("=== Q1: Top Completed Projects Over $10M ===")
print(q1)

q2 = pd.read_sql_query("""
    SELECT
        sector,
        COUNT(*)                          AS total_projects,
        ROUND(AVG(project_value_cad), 0)  AS avg_value_cad,
        ROUND(AVG(delay_days), 1)         AS avg_delay_days,
        SUM(safety_incidents)             AS total_incidents
    FROM projects
    GROUP BY sector
    ORDER BY avg_value_cad DESC
""", conn)
print("\n=== Q2: Summary by Sector ===")
print(q2)

q3 = pd.read_sql_query("""
    SELECT
        p.project_id,
        r.city,
        r.province,
        c.contractor_name,
        p.sector,
        ROUND(p.project_value_cad / 1000000, 2) AS value_millions,
        p.status,
        p.delay_days
    FROM projects p
    JOIN regions     r ON p.region_id     = r.region_id
    JOIN contractors c ON p.contractor_id = c.contractor_id
    ORDER BY p.project_value_cad DESC
    LIMIT 15
""", conn)
print("\n=== Q3: Projects with City and Contractor (JOIN) ===")
print(q3)

q4 = pd.read_sql_query("""
    SELECT
        r.city,
        COUNT(p.project_id)          AS total_projects,
        ROUND(AVG(p.delay_days), 1)  AS avg_delay_days,
        MAX(p.delay_days)            AS worst_delay
    FROM projects p
    JOIN regions r ON p.region_id = r.region_id
    GROUP BY r.city
    ORDER BY avg_delay_days DESC
""", conn)
print("\n=== Q4: Average Delay by City ===")
print(q4)

conn.close()


=== Q1: Top Completed Projects Over $10M ===
     project_id          sector  value_millions     status  delay_days
0  AB-2023-0043      Industrial           43.80  Completed       165.0
1  AB-2021-0153       Oil & Gas           41.97  Completed        78.0
2  AB-2021-0021      Industrial           41.45  Completed       107.0
3  AB-2022-0086      Commercial           40.46  Completed       179.0
4  AB-2022-0150      Industrial           38.69  Completed         3.0
5  AB-2021-0121  Infrastructure           38.29  Completed       138.0
6  AB-2023-0171       Oil & Gas           37.86  Completed       132.0
7  AB-2022-0042     Residential           37.15  Completed        52.0
8  AB-2022-0162      Commercial           36.31  Completed       126.0
9  AB-2022-0130      Industrial           34.79  Completed       116.0

=== Q2: Summary by Sector ===
           sector  total_projects  avg_value_cad  avg_delay_days  \
0     Residential              38     23043188.0            82.5   
1  Infr